In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('data\stud.csv')

In [3]:
df.head(3)

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93


In [4]:
x = df.drop('math_score',axis = 1)
y = df['math_score']

In [5]:
categorical_columns = x.select_dtypes(include='object').columns
numerical_columns = x.select_dtypes(exclude='object').columns

In [6]:
for feature in df.columns:
  if df[feature].dtype == 'O':
    print(f"Unique features in {feature} : {df[feature].unique()}")

Unique features in gender : ['female' 'male']
Unique features in race_ethnicity : ['group B' 'group C' 'group A' 'group D' 'group E']
Unique features in parental_level_of_education : ["bachelor's degree" 'some college' "master's degree" "associate's degree"
 'high school' 'some high school']
Unique features in lunch : ['standard' 'free/reduced']
Unique features in test_preparation_course : ['none' 'completed']


In [7]:
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
  [("StandardScaler",numerical_transformer,numerical_columns),
   ("OneHotEncoder",categorical_transformer,categorical_columns)]
)


In [8]:
x = preprocessor.fit_transform(x)

In [9]:
x.shape

(1000, 19)

In [10]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.20,random_state=42)

In [11]:
models = {
  "Linear Regression":LinearRegression(),
  "AdaBoost Regressor":AdaBoostRegressor(),
  "DecisionTree Regressor":DecisionTreeRegressor(),
  "CatBoost Regressor":CatBoostRegressor(),
  "RandomForest Regressor":RandomForestRegressor(),
  "KNeighbors Regressor":KNeighborsRegressor(),
  "SVR":SVR(),
  "XGB Regressor":XGBRegressor(),
  "Ridge":Ridge(),
  "Lasso":Lasso()
}

In [12]:
def scores(true,prediction):
  mse = mean_squared_error(true,prediction)
  mae = mean_absolute_error(true,prediction)
  r2 = r2_score(true,prediction)
  rmse = np.sqrt(mse)

  return mse,mae,r2,rmse

In [13]:
model_list = []
r2_list = []
for i in range(len(list(models))):
  model = list(models.values())[i]
  model.fit(x_train,y_train)

  y_train_pred = model.predict(x_train)
  y_test_pred = model.predict(x_test)

  train_mse,train_mae,train_r2,train_rmse = scores(y_train,y_train_pred)
  test_mse,test_mae,test_r2,test_rmse = scores(y_test,y_test_pred)

  model_list.append(list(models.keys())[i])
  r2_list.append(test_r2)

  print(list(models.keys())[i])
  print("-"*50)
  print("Training model prediction scores")
  print()
  print(f"Training model mean squared error score : {train_mse}")
  print(f"Training model root mean squared error score : {train_rmse}")
  print(f"Training model mean absolute error score : {train_mae}")
  print(f"Training model mean absolute error score : {train_r2}")
  print("-"*50)
  print("Test model prediction scores")
  print()
  print(f"Test model mean squared error score : {test_mse}")
  print(f"Test model root mean squared error score : {test_rmse}")
  print(f"Test model mean absolute error score : {test_mae}")
  print(f"Test model mean absolute error score : {test_r2}")
  print("="*50)
  

Linear Regression
--------------------------------------------------
Training model prediction scores

Training model mean squared error score : 28.33487038064859
Training model root mean squared error score : 5.323050852720514
Training model mean absolute error score : 4.266711846071956
Training model mean absolute error score : 0.8743172040139593
--------------------------------------------------
Test model prediction scores

Test model mean squared error score : 29.095169866715466
Test model root mean squared error score : 5.3939938697328405
Test model mean absolute error score : 4.214763142474849
Test model mean absolute error score : 0.8804332983749565
AdaBoost Regressor
--------------------------------------------------
Training model prediction scores

Training model mean squared error score : 33.98307678759046
Training model root mean squared error score : 5.829500560733352
Training model mean absolute error score : 4.76961539788581
Training model mean absolute error score : 0.

  File "c:\ML_Project\venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\ML_Project\venv\lib\subprocess.py", line 489, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\ML_Project\venv\lib\subprocess.py", line 854, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\ML_Project\venv\lib\subprocess.py", line 1307, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


KNeighbors Regressor
--------------------------------------------------
Training model prediction scores

Training model mean squared error score : 32.635549999999995
Training model root mean squared error score : 5.712753276660913
Training model mean absolute error score : 4.51675
Training model mean absolute error score : 0.8552410116072555
--------------------------------------------------
Test model prediction scores

Test model mean squared error score : 52.617
Test model root mean squared error score : 7.253757646902741
Test model mean absolute error score : 5.619
Test model mean absolute error score : 0.7837702557426202
SVR
--------------------------------------------------
Training model prediction scores

Training model mean squared error score : 43.257024268031365
Training model root mean squared error score : 6.57700724251018
Training model mean absolute error score : 4.869189452384867
Training model mean absolute error score : 0.8081281585902299
----------------------------

In [14]:
pd.DataFrame(list(zip(model_list,r2_list)),columns=["Models","R2_score"]).sort_values(by=["R2_score"])

,Models,R2_score
6,SVR,0.728600
2,DecisionTree Regressor,0.733396
5,KNeighbors Regressor,0.783770
7,XGB Regressor,0.821220
9,Lasso,0.825320
3,CatBoost Regressor,0.851632
4,RandomForest Regressor,0.853581
1,AdaBoost Regressor,0.853749
0,Linear Regression,0.880433
8,Ridge,0.880593
